# Content-Based Recommendation System

## Objective

In this notebook a baseline content-based recommendation system is developed for classical music works.

The goal is to recommend musicality similar works based on the extracted metadata features rather than user interactions. Each composition is represented using its musical characteristics and similar works are identified based on the similarity between their feature representations.

The features used for measuring similarity include:
- **Epoch**: historical period of composition
- **Form**: structural characteristics of the work
- **Extended Genre**: detailed work categories extracted from metadata
- **Instrumentation**: instruments and ensembles associated with the work
- **Mode**: tonal characteristics such as major/minor information
- **Tempo Marking**: tempo-related indications when available

A content-based approach is selected as the baseline because the dataset does not contain user ratings or interaction history. This allows recommendations to be generated using only the musical properties of each work.

This workflow consists of:
1. Preparing and combining musical features
2. Transforming categorical metadata into numerical representations
3. Calculating similarity between works
4. Generating recommendations based on the most similar compositions

This baseline model will later serve as a foundation for improvements using techniques such as embeddings and hybrid recommendation approaches.

In [1]:
import pandas as pd
import joblib

from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
df = pd.read_csv(Path('../data/processed/classical_music_feature_engineered.csv'))

In [3]:
df.head()

,title,subtitle,popular,recommended,work_id,genre,composer_id,composer_name,birth,death,epoch,title_subtitle,form,extended_genre,instrumentation,mode,tempo_marking
0,China Gates,NaN,1,1,16893,Keyboard,149,John Adams,1947,NaN,21st Century,China Gates,NaN,Keyboard,Piano,NaN,NaN
1,Doctor Atomic Symphony,NaN,0,1,16878,Orchestral,149,John Adams,1947,NaN,21st Century,Doctor Atomic Symphony,Symphony,Orchestral,Orchestra,NaN,NaN
2,Harmonielehre,NaN,1,1,16875,Orchestral,149,John Adams,1947,NaN,21st Century,Harmonielehre,NaN,Orchestral,Orchestra,NaN,NaN
3,Nixon in China,Opera,1,1,16890,Stage,149,John Adams,1947,NaN,21st Century,Nixon in China Opera,NaN,Opera,"Voice, Chorus, Orchestra",NaN,NaN
4,Short Ride in a Fast Machine,NaN,1,1,16896,Orchestral,149,John Adams,1947,NaN,21st Century,Short Ride in a Fast Machine,NaN,Orchestral,Orchestra,NaN,NaN


In [4]:
df.columns

Index(['title', 'subtitle', 'popular', 'recommended', 'work_id', 'genre',
       'composer_id', 'composer_name', 'birth', 'death', 'epoch',
       'title_subtitle', 'form', 'extended_genre', 'instrumentation', 'mode',
       'tempo_marking'],
      dtype='str')

In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 13014 entries, 0 to 13013
Data columns (total 17 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   title            13014 non-null  str    
 1   subtitle         1288 non-null   str    
 2   popular          13014 non-null  int64  
 3   recommended      13014 non-null  int64  
 4   work_id          13014 non-null  int64  
 5   genre            13014 non-null  str    
 6   composer_id      13014 non-null  int64  
 7   composer_name    13014 non-null  str    
 8   birth            13014 non-null  int64  
 9   death            12747 non-null  float64
 10  epoch            13014 non-null  str    
 11  title_subtitle   13014 non-null  str    
 12  form             4901 non-null   str    
 13  extended_genre   13014 non-null  str    
 14  instrumentation  12746 non-null  str    
 15  mode             3597 non-null   str    
 16  tempo_marking    204 non-null    str    
dtypes: float64(1), int64(5)

### Feature Representation

Since the extracted musical features are mostly categorical, they need to be transformed into a numerical representation that can be used to measure similarity between works.

For the baseline model, the selected metadata features are combined into a single text representation for each composition. This allows TF-IDF to capture the importance of different musical characteristics and represent each work as a numerical vector.

The following features are used:
- Epoch
- Form
- Extended Genre
- Instrumentation
- Mode
- Tempo markings

Composer information is excluded from the similarity calculation to avoid recommendations being limited only to works by the same composer.

In [6]:
df['instrumentation'] = df['instrumentation'].str.replace(',', ' ', regex=False).str.strip()

In [7]:
features = ['epoch', 'form', 'extended_genre', 'instrumentation', 'mode', 'tempo_marking']

In [8]:
df['combined_features'] = (
    df[features]
    .fillna('')
    .astype(str)
    .agg(" ".join, axis=1)
)

In [9]:
df[['title', 'combined_features']].sample(10)

,title,combined_features
12769,Trumpet Concerto in C,Baroque Concerto Orchestral Orchestra
2582,Si j’ay perdu mon amy,Renaissance Vocal Voice
12077,Il brigidino,Romantic Song Voice Piano
4211,"Divertimento in F major, for oboe, bassoon, 2 ...",Classical Chamber Violin Viola Oboe Major
12950,"Polka, WWV 84",Romantic Polka Keyboard Piano
7248,"Müsst ich auch durch tausend Drachen, K.435",Classical Aria Orchestra
1398,"Symphony no. 2 in D major, op. 36",Early Romantic Symphony Orchestral Orchestra M...
9599,"Sonata in D major, K.346, L.60",Baroque Sonata Keyboard Piano Major
6426,Violin Sonata in F,Romantic Sonata Chamber Violin
3719,"Aci, Galatea e Polifemo, HWV.72",Baroque Vocal Voice


### TF-IDF Feature Transformation

The combined musical metadata is transformed into numerical vectors using TF-IDF (Term Frequency-Inverse Document Frequency).

TF-IDF assigns higher importance to features that are frequent within a specific work but less common across the entire dataset. This allows the model to represent each composition based on its distinctive musical characteristics.

Each work is converted into a vector representation, which enables measuring similarity between compositions using mathematical distance metrics.

In [10]:
tfidf = TfidfVectorizer()

In [11]:
tfidf_matrix = tfidf.fit_transform(df['combined_features'])

In [12]:
tfidf_matrix.shape

(13014, 106)

In [13]:
tfidf.get_feature_names_out()[:20]

array(['20th', '21st', 'adagio', 'allegretto', 'allegro', 'andante',
       'arabesque', 'aria', 'ballade', 'ballet', 'baroque', 'brass',
       'canon', 'cantata', 'capriccio', 'cello', 'century', 'chaconne',
       'chamber', 'chamberensemble'], dtype=object)

### Similarity Calculation

After transforming each composition into a TF-IDF vector representation, the similarity between works can be measured.

Cosine similarity is used to compare the angle between two feature vectors. A higher cosine similarity value indicates that two compositions share more similar musical characteristics based on their extracted metadata.

The resulting similarity matrix contains the similarity score between every pair of works and is used to retrieve the most similar compositions for recommendation.

In [14]:
similarity_matrix = cosine_similarity(tfidf_matrix)

In [15]:
similarity_matrix.shape

(13014, 13014)

In [16]:
def recommend(title, composer, n=5):
    idx = df[(df['title'] == title) & (df['composer_name'] == composer)].index[0]
    similarity_scores = list(enumerate(similarity_matrix[idx]))
    similarity_scores = sorted(similarity_scores, key=lambda x: x[1], reverse=True)
    similarity_scores = similarity_scores[1:n+1]

    recommended_indices = [i[0] for i in similarity_scores]
    return df.loc[recommended_indices, ['title', 'composer_name','epoch', 'form', 'extended_genre', 'instrumentation', 'mode', 'tempo_marking']]

In [17]:
recommend('Tempo di valse in A major, JS 2', 'Jean Sibelius')

,title,composer_name,epoch,form,extended_genre,instrumentation,mode,tempo_marking
11730,"Valse à Betsy Lerche in A flat major, JS 1",Jean Sibelius,Late Romantic,Waltz,Keyboard,Piano,Major,NaN
11737,Waltz in D flat major,Jean Sibelius,Late Romantic,Waltz,Keyboard,Piano,Major,NaN
2467,"Valse romantique, L.71",Claude Debussy,Late Romantic,Waltz,Keyboard,Piano,NaN,NaN
8809,8 Valses nobles et sentimentales,Maurice Ravel,Late Romantic,Waltz,Keyboard,Piano,NaN,NaN
11728,"Valse lyrique, op. 96a",Jean Sibelius,Late Romantic,Waltz,Keyboard,Piano,NaN,NaN


In [18]:
recommend('8 Valses nobles et sentimentales', 'Maurice Ravel')

,title,composer_name,epoch,form,extended_genre,instrumentation,mode,tempo_marking
8809,8 Valses nobles et sentimentales,Maurice Ravel,Late Romantic,Waltz,Keyboard,Piano,NaN,NaN
11728,"Valse lyrique, op. 96a",Jean Sibelius,Late Romantic,Waltz,Keyboard,Piano,NaN,NaN
11729,"Valse triste, op. 44",Jean Sibelius,Late Romantic,Waltz,Keyboard,Piano,NaN,NaN
11739,Waltz in E,Jean Sibelius,Late Romantic,Waltz,Keyboard,Piano,NaN,NaN
11691,"Tempo di valse in A major, JS 2",Jean Sibelius,Late Romantic,Waltz,Keyboard,Piano,Major,NaN


In [19]:
recommend('Symphony in B minor', 'Claude Debussy')

,title,composer_name,epoch,form,extended_genre,instrumentation,mode,tempo_marking
2890,"Falstaff, symphonic study in C minor, op. 68",Edward Elgar,Late Romantic,Symphony,Orchestral,Orchestra,Minor,NaN
5933,"Symphony no. 2 in C minor, ""Resurrection""",Gustav Mahler,Late Romantic,Symphony,Orchestral,Orchestra,Minor,NaN
5934,Symphony no. 5 in C sharp minor,Gustav Mahler,Late Romantic,Symphony,Orchestral,Orchestra,Minor,NaN
5935,Symphony no. 6 in A minor,Gustav Mahler,Late Romantic,Symphony,Orchestral,Orchestra,Minor,NaN
5948,Symphony no. 3 in D minor,Gustav Mahler,Late Romantic,Symphony,Orchestral,Orchestra,Minor,NaN


## Baseline Recommendation Evaluation

The baseline content-based recommendation system was evaluated using qualitative examples from the dataset.

The generated recommendations were manually inspected to determine whether the retrieved works shared meaningful musical characteristics with the input composition.

The results show that the model is able to identify similar works based on shared metadata features, including:

* Historical period (epoch)
* Musical form
* Genre
* Performing forces (instrumentation)
* Mode and tempo information when available

For example, when a late Romantic piano waltz was used as input, the system retrieved other works sharing characteristics such as the same historical period, waltz form, and piano instrumentation, including works by different composers.

Since composer information was not included in the similarity calculation, the recommendations are based on musical attributes rather than composer identity.

These results indicate that the extracted metadata features provide a meaningful representation of musical similarity and can serve as a foundation for future improvements.


### Save Model Artifacts

The trained TF-IDF vectorizer and similarity matrix are saved to reuse the recommendation pipeline without recomputing the model.

In [20]:
joblib.dump(similarity_matrix, Path('../models/similarity_matrix.pkl'))

['../models/similarity_matrix.pkl']

# Further Model Improvements

The baseline recommendation system relies solely on structured metadata extracted from the Open Opus dataset. While this approach provides meaningful recommendations it has several limitations.

First, recommendations can only be generated for works that already exist in the dataset. Second, musical similarity is limited to the available metadata and cannot capture higher-level concepts such as mood, expression or stylistic characteristics.

The proposed improvements include:

1. Handling incomplete queries from user
2. Semantic embeddings to capture relationships beyond manually extracted metadata.
3. A hybrid recommendation system that combines multiple similarity sources.